# 01b Cleaning timing: CPU vs GPU
- **Owner:** Varuna
- **Purpose:** Times each step of the cleaning pipeline in `01_cleaning.ipynb` so the same steps can be compared on CPU and GPU.
- **How to run on CPU:** leave the `%load_ext cudf.pandas` line commented out, set `MODE = "CPU"`, run all cells.
- **How to run on GPU (Colab, GPU runtime):** uncomment the `%load_ext cudf.pandas` line, set `MODE = "GPU"`, update `DATA_PATH` and `OUT_DIR` to the Colab file locations, run all cells.
- **Fair comparison:** run CPU and GPU in the same Colab session. Run the timing cell twice and report the second run, because the first run includes start-up overhead.
- **Output:** a timing table saved as `docs/timing_cleaning_cpu.csv` or `docs/timing_cleaning_gpu.csv`.

In [1]:
# GPU switch: must run BEFORE pandas is imported. Leave commented for CPU.
# %load_ext cudf.pandas

import os, time, platform
from contextlib import contextmanager
import pandas as pd
import numpy as np

MODE = "CPU"                                   # change to "GPU" on the GPU run
DATA_PATH = "../data/listings.csv.gz"          # in Colab: "listings.csv.gz"
OUT_DIR = "../docs"                            # in Colab: "."
OUT_CSV = "../data/listings_clean_timing.csv"  # in Colab: "listings_clean_timing.csv"

@contextmanager
def timer(label):
    t0 = time.perf_counter()
    yield
    times.append({"Step": label, "Seconds": round(time.perf_counter() - t0, 3)})

In [2]:
times = []

with timer("Load CSV"):
    df = pd.read_csv(DATA_PATH)

with timer("Summary statistics (describe)"):
    _ = df.describe()

with timer("Column audit (missing %, unique counts)"):
    audit = pd.DataFrame({"missing_pct": df.isna().mean() * 100, "n_unique": df.nunique()})

with timer("Drop empty and metadata columns"):
    empty_cols = audit.index[audit["missing_pct"] == 100].tolist()
    meta_cols = [c for c in df.columns if c not in empty_cols and
                 (c.endswith("_url") or c in ["scrape_id", "last_scraped", "source", "calendar_last_scraped"])]
    clean = df.drop(columns=empty_cols + meta_cols)

with timer("Text placeholders to missing"):
    stand_ins = ["N/A", "NA", "null", "?", "-", "unknown", ""]
    for c in ["description", "host_about"]:
        clean[c] = clean[c].where(~clean[c].isin(stand_ins))

with timer("Price: text to number, cap at 99th percentile"):
    clean["price"] = pd.to_numeric(clean["price"].str.replace(r"[$,]", "", regex=True), errors="coerce")
    clean.loc[clean["price"] <= 0, "price"] = np.nan
    cap = clean["price"].quantile(0.99)
    clean["price_capped"] = clean["price"].clip(upper=cap)

with timer("Recover bathrooms from text (regex)"):
    bt = clean["bathrooms_text"]
    num = pd.to_numeric(bt.str.extract(r"(\d+\.?\d*)")[0], errors="coerce")
    num = num.mask(num.isna() & bt.str.contains("half", case=False, na=False), 0.5)
    fill = clean["bathrooms"].isna() & num.notna()
    clean.loc[fill, "bathrooms"] = num[fill]

with timer("Duplicate check (without id)"):
    dup = clean.drop(columns=["id"]).duplicated()
    clean = clean[~dup]

with timer("Target: zero ratings to missing, create top_rated"):
    clean.loc[clean["review_scores_rating"] == 0, "review_scores_rating"] = np.nan
    rated = clean["review_scores_rating"].notna()
    clean["top_rated"] = (clean["review_scores_rating"] >= 4.8).astype("Int64").where(rated)

with timer("Size checks, flags, and night caps"):
    clean.loc[clean["beds"] == 0, "beds"] = np.nan
    size = ["bedrooms", "beds", "bathrooms"]
    p999 = clean[size].quantile(0.999)
    clean["size_extreme"] = (clean[size] > p999).any(axis=1).astype(int)
    clean["minimum_nights_capped"] = clean["minimum_nights"].clip(upper=365)
    clean["maximum_nights_capped"] = clean["maximum_nights"].clip(upper=1125)

with timer("Fill missing with room-type median (group by)"):
    for c in ["price_capped", "beds", "bedrooms", "bathrooms"]:
        clean[f"{c}_missing"] = clean[c].isna().astype(int)
        med = clean.groupby("room_type")[c].transform("median")
        clean[f"{c}_imp"] = clean[c].fillna(med).fillna(clean[c].median())

with timer("Save cleaned CSV"):
    clean.to_csv(OUT_CSV, index=False)

print("Final shape:", clean.shape)

Final shape: (92635, 80)


In [3]:
res = pd.DataFrame(times)
res.loc[len(res)] = {"Step": "Total", "Seconds": round(res["Seconds"].sum(), 3)}
res["Mode"] = MODE
res["Machine"] = platform.platform()
os.makedirs(OUT_DIR, exist_ok=True)
res.to_csv(f"{OUT_DIR}/timing_cleaning_{MODE.lower()}.csv", index=False)
display(res)

,Step,Seconds,Mode,Machine
0,Load CSV,1.552,CPU,macOS-26.6.2-arm64-arm-64bit
1,Summary statistics (describe),0.083,CPU,macOS-26.6.2-arm64-arm-64bit
2,"Column audit (missing %, unique counts)",0.195,CPU,macOS-26.6.2-arm64-arm-64bit
3,Drop empty and metadata columns,0.000,CPU,macOS-26.6.2-arm64-arm-64bit
4,Text placeholders to missing,0.004,CPU,macOS-26.6.2-arm64-arm-64bit
5,"Price: text to number, cap at 99th percentile",0.017,CPU,macOS-26.6.2-arm64-arm-64bit
6,Recover bathrooms from text (regex),0.050,CPU,macOS-26.6.2-arm64-arm-64bit
7,Duplicate check (without id),0.172,CPU,macOS-26.6.2-arm64-arm-64bit
8,"Target: zero ratings to missing, create top_rated",0.001,CPU,macOS-26.6.2-arm64-arm-64bit
9,"Size checks, flags, and night caps",0.003,CPU,macOS-26.6.2-arm64-arm-64bit
